In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [3]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [1]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import AdvancedLSTM
import json

In [2]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [3]:
X_train = torch.load(file_path + "/X_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
X_val = torch.load(file_path + "/X_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train.shape}, Targets: {y_train.shape}")

Train sequences: torch.Size([72339, 5, 38]), Targets: torch.Size([72339, 1])


In [4]:
player_ids_train = torch.load(file_path + "/train_player_ids.pt", weights_only=True)
player_ids_val = torch.load(file_path + "/val_player_ids.pt", weights_only=True)
print(f"Player IDs Train: {player_ids_train.shape}, Val: {player_ids_val.shape}")

Player IDs Train: torch.Size([72339]), Val: torch.Size([11384])


In [5]:
with open(f"{file_path}/vocab/player_name_to_idx.json") as f:
    name_to_idx = json.load(f)
with open(f"{file_path}/vocab/unk_id.txt") as f:
    unk_id = int(f.read())

In [6]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train, y_train, X_val, y_val,
                                    player_ids_train=player_ids_train, player_ids_val=player_ids_val,
                                    epochs=1, n_trials=1, transform=False, num_workers=4,
                                    player_vocab_size=len(name_to_idx), player_embed_dim=32,
                                    unknown_player_index=unk_id)

# Full training with best hyperparameters
print("\nTraining final model with best hyperparameters...")
adv_model = AdvancedLSTM(input_dim=X_train.shape[-1], hidden_dim=best_params['hidden_dim'],
                            output_dim=1, num_layers=best_params['num_layers'],
                            dropout=best_params['dropout'], num_fc_layers=best_params['num_fc_layers'],
                                        player_vocab_size=len(name_to_idx), player_embed_dim=32,
                                        unknown_player_index=unk_id)


train_model(
    adv_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    player_ids_train=player_ids_train,
    player_ids_val=player_ids_val,
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    batch_size=best_params['batch_size'],
    epochs=100,  # Full training
    verbose=2,
    transform=False,
    num_workers=4,
)

Running random search with 1 trials...

Trial 1/1
Params: {'learning_rate': 0.0005, 'hidden_dim': 64, 'weight_decay': 0.001, 'num_layers': 1, 'dropout': 0.1, 'num_fc_layers': 3, 'batch_size': 256}


/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(
/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(


New best RMSE: 3.4754

Top 5 hyperparameter combinations:
1. RMSE: 3.4754, Params: {'learning_rate': 0.0005, 'hidden_dim': 64, 'weight_decay': 0.001, 'num_layers': 1, 'dropout': 0.1, 'num_fc_layers': 3, 'batch_size': 256, 'rmse': np.float64(3.4754462164405093)}

Best hyperparameters: {'learning_rate': 0.0005, 'hidden_dim': 64, 'weight_decay': 0.001, 'num_layers': 1, 'dropout': 0.1, 'num_fc_layers': 3, 'batch_size': 256}
Best RMSE: 3.4754

Training final model with best hyperparameters...


Epoch  1/100: 100%|██████████| 283/283 [00:12<00:00, 22.08it/s, loss=21.5305, lr=2.13e-5]

Epoch 1 Training MSE: 18.2947


Epoch 1 validation RMSE: 4.0250
Epoch 1 validation MAE: 2.8061
Best model saved at epoch 1 with RMSE: 4.0250


Epoch  2/100:  20%|██        | 58/283 [00:03<00:09, 24.44it/s, loss=14.5929, lr=2.19e-5]libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x1202a1b20>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1568, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/multiprocessing/process.py"

KeyboardInterrupt: 